# Data Cleaning Step

In [12]:
import pandas as pd, geopandas as gpd
from shapely import wkt
from shapely.validation import make_valid
import pandas as pd, geopandas as gpd, folium
import folium
import zipfile
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import io, os, re, time
import numpy as np
import requests
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

## Fault Lines 
Finds where all the fault lines lie. Could be added with a dataset where we can see how far the effects of an earthquake reaches.
https://data.bayareametro.gov/Land-People/Seismic-Hazard-Zone-Alquist-Priolo-HESS-/7fdj-fhcc/about_data

In [13]:
!pip -q install geopandas shapely folium mapclassify

df = pd.read_csv("data/Seismic_Hazard_Zone_(Alquist-Priolo)_(HESS)_20261026.csv")

df = df.replace(r"^\s*$", pd.NA, regex=True)

for c in ["map_releas", "zn_release"]:
    df[c] = pd.to_datetime(df[c])
for c in ["map_revise", "zn_revised"]:
    df[c] = df[c].map({"Y": True, "N": False})
for c in ["prev_map_r", "prev_zn_re"]:
    df[c] = df[c].str.replace(r"\s*;\s*", ";", regex=True)

df = df.drop(columns=["Shape__Area", "Shape__Length", "Shape_Area", "Shape_Length"])

df = df.rename(columns={
    "map_releas": "map_released", "map_revise": "map_revised", "prev_map_r": "prev_map_releases",
    "zn_release": "zone_released", "zn_revised": "zone_revised", "prev_zn_re": "prev_zone_releases",
    "geopdflink": "map_pdf_link", "reportlink": "report_link",
})

gdf = gpd.GeoDataFrame(df, geometry=df["geometry"].apply(wkt.loads), crs="EPSG:4326")
gdf["geometry"] = gdf.geometry.apply(make_valid)

gdf = gdf.to_crs("EPSG:3310")
gdf["area_m2"] = gdf.area

print(gdf.shape)                                    # expect (133, 13)
print("all valid:", gdf.is_valid.all())
print("duplicate shapes:", gdf.geometry.to_wkb().duplicated().sum())
print("total km²:", round(gdf["area_m2"].sum() / 1e6, 1))  # ~469
gdf.dtypes

gdf.to_file("data/ap_fault_zones_clean.gpkg", driver="GPKG")

(133, 13)
all valid: True
duplicate shapes: 0
total km²: 468.9


## Historic Earthquakes
https://www.usgs.gov/maps/earthquakes-and-faults-san-francisco-bay-area-1970-2003

In [14]:
target_file = "data/bayarea_earthquakes (1).csv"

print(f"Reading and cleaning: {target_file}")
df = pd.read_csv(target_file)
df.columns = df.columns.str.lower().str.strip()

lat_col = 'latitude (n)'
lon_col = 'longitude (w)'

# Clean coordinates
df = df.dropna(subset=[lat_col, lon_col])
df[lat_col] = pd.to_numeric(df[lat_col], errors='coerce')
df[lon_col] = pd.to_numeric(df[lon_col], errors='coerce')
df = df.dropna(subset=[lat_col, lon_col])

# Convert positive longitude (W) to negative degrees decimal for correct mapping
df[lon_col] = -df[lon_col].abs()

# Convert date to datetime
if 'date' in df.columns:
    df['date'] = pd.to_datetime(df['date'], errors='coerce')

# Convert to GeoDataFrame
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df[lon_col], df[lat_col]),
    crs="EPSG:4326"
)

    # Reproject to California Albers
gdf_projected = gdf.to_crs("EPSG:3310")
print(f"Successfully loaded and cleaned {len(gdf_projected)} earthquake records!")

# Save cleaned dataset
gdf_projected.to_file("data/sfba_earthquakes_clean.gpkg", driver="GPKG")
print("Saved cleaned dataset as 'sfba_earthquakes_clean.gpkg'")

Reading and cleaning: data/bayarea_earthquakes (1).csv
Successfully loaded and cleaned 22 earthquake records!
Saved cleaned dataset as 'sfba_earthquakes_clean.gpkg'


C:\Users\katie\AppData\Local\Temp\ipykernel_60588\3992400318.py:21: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['date'] = pd.to_datetime(df['date'], errors='coerce')


## HCD APR Data
https://www.huduser.gov/portal/datasets/cp.html#data_2006-2022

In [15]:
zip_path = "data/2018thru2022-050-csv (1).zip"

with zipfile.ZipFile(zip_path, "r") as z:
    print(z.namelist())

zip_path = "data/2018thru2022-050-csv (1).zip"

with zipfile.ZipFile(zip_path, "r") as z:
    with z.open("050/Table1.csv") as f:
        income = pd.read_csv(f, encoding="latin1")
zip_path = "data/2018thru2022-050-csv (1).zip"

with zipfile.ZipFile(zip_path, "r") as z:
    with z.open("050/Table1.csv") as f:
        Household = pd.read_csv(f, encoding="latin1")

income.head()
Household.select_dtypes(include="object").columns.tolist()
Household.isna().sum()[Household.isna().sum() > 0]
Household = Household.rename(columns={
    "source": "Year",
    "sumlevel": "Geography_Level",
    "geoid": "County_ID",
    "name": "County",
    "st": "State_Code",
    "cnty": "County_Code"
})

bay_area = [
    "Alameda County, California",
    "Contra Costa County, California",
    "Marin County, California",
    "Napa County, California",
    "San Francisco County, California",
    "San Mateo County, California",
    "Santa Clara County, California",
    "Solano County, California",
    "Sonoma County, California"
]

bay_household = Household[Household["County"].isin(bay_area)].copy()

with zipfile.ZipFile(zip_path, "r") as z:
    with z.open("050/Table8.csv") as f:
        table8 = pd.read_csv(f, encoding="latin1")

bay8 = table8[table8["name"].isin(bay_area)].copy()

bay8["Below 50%"] = bay8["T8_est3"] + bay8["T8_est16"] + bay8["T8_est69"] + bay8["T8_est82"]
bay8["50-80%"] = bay8["T8_est29"] + bay8["T8_est95"]
bay8["80-100%"] = bay8["T8_est42"] + bay8["T8_est108"]
bay8["Above 100%"] = bay8["T8_est55"] + bay8["T8_est121"]

income_cols = ["Below 50%", "50-80%", "80-100%", "Above 100%"]

county_income = bay8.set_index("name")[income_cols]
county_income.index = county_income.index.str.replace(" County, California", "", regex=False)
county_income = county_income.div(county_income.sum(axis=1), axis=0) * 100
check = bay8[["name", "T8_est1"]].copy()
check["Calculated_Total"] = bay8[income_cols].sum(axis=1)
check["Difference"] = check["Calculated_Total"] - check["T8_est1"]
county_income.to_csv("data/county_income.csv")

['050/', '050/Table1.csv', '050/Table10.csv', '050/Table11.csv', '050/Table12.csv', '050/Table13.csv', '050/Table14A.csv', '050/Table14B.csv', '050/Table15A.csv', '050/Table15B.csv', '050/Table15C.csv', '050/Table16.csv', '050/Table17A.csv', '050/Table17B.csv', '050/Table18A.csv', '050/Table18B.csv', '050/Table18C.csv', '050/Table2.csv', '050/Table3.csv', '050/Table4.csv', '050/Table5.csv', '050/Table6.csv', '050/Table7.csv', '050/Table8.csv', '050/Table9.csv']


C:\Users\katie\AppData\Local\Temp\ipykernel_60588\809203509.py:18: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  Household.select_dtypes(include="object").columns.tolist()


## CTAC Data for State Low Income tax Credits
https://www.treasurer.ca.gov/ctcac/projects

In [ ]:
filename = "data/List of Projects_1.xlsx"

workbook = pd.ExcelFile(filename)
print("Sheets:", workbook.sheet_names)

ctcac = pd.read_excel(filename, sheet_name=0)
ctcac = pd.read_excel(filename, sheet_name="California Mapped Projects")

# The 9 Bay Area counties
bay_counties = [
    "Alameda", "Contra Costa", "Marin", "Napa", "San Francisco",
    "San Mateo", "Santa Clara", "Solano", "Sonoma"
]

projects = ctcac.copy()

projects["Project County"] = projects["Project County"].fillna("").astype(str).str.strip()
projects["Project City"] = projects["Project City"].fillna("").astype(str).str.strip()

# Keep projects in the 9 counties
bay_projects = projects[
    projects["Project County"].str.casefold().isin(
        [county.casefold() for county in bay_counties]
    )
].copy()

bay_projects["Low Income Units"] = pd.to_numeric(
    bay_projects["Low Income Units"], errors="coerce"
)

print("Bay Area project rows:", len(bay_projects))
print("Missing unit counts:", bay_projects["Low Income Units"].isna().sum())


city_units = (
    bay_projects.groupby(["Project County", "Project City"])["Low Income Units"]
    .sum(min_count=1)
    .reset_index()
    .sort_values("Low Income Units", ascending=False)
)
# Matching  CTCAC abriv to the Census place name
city_units["Project City"] = city_units["Project City"].replace({
    "Pt. Reyes Station": "Point Reyes Station"
})
# Download Census city boundaries and county boundaries for California
cities_url = (
    "https://www2.census.gov/geo/tiger/GENZ2023/shp/"
    "cb_2023_06_place_500k.zip"
)
counties_url = (
    "https://www2.census.gov/geo/tiger/GENZ2023/shp/"
    "cb_2023_us_county_500k.zip"
)

cities = gpd.read_file(cities_url).to_crs("EPSG:3310")
counties = gpd.read_file(counties_url).to_crs(cities.crs)

# Keep the nine Bay Area counties
bay_boundary = counties[counties["NAME"].isin(bay_counties)]

# Keep city boundaries that intersect the Bay Area
bay_cities = gpd.clip(cities, bay_boundary)


totals = city_units.copy()
totals["city_key"] = totals["Project City"].str.strip().str.casefold()
totals = totals.groupby("city_key")["Low Income Units"].sum(min_count=1)

bay_cities["city_key"] = bay_cities["NAME"].str.strip().str.casefold()
bay_cities = bay_cities.merge(
    totals, left_on="city_key", right_index=True, how="left"
)


Sheets: ['Instructions', 'California Mapped Projects', 'Alameda', 'Alpine', 'Amador', 'Butte', 'Calaveras', 'Colusa', 'Contra Costa', 'Del Norte', 'El Dorado', 'Fresno', 'Glenn', 'Humboldt', 'Imperial', 'Inyo', 'Kern', 'Kings', 'Lake', 'Lassen', 'Los Angeles', 'Madera', 'Marin', 'Mariposa', 'Mendocino', 'Merced', 'Modoc', 'Mono', 'Monterey', 'Napa', 'Nevada', 'Orange', 'Placer', 'Plumas', 'Riverside', 'Sacramento', 'San Benito', 'San Bernardino', 'San Diego', 'San Francisco', 'San Joaquin', 'San Luis Obispo', 'San Mateo', 'Santa Barbara', 'Santa Clara', 'Santa Cruz', 'Shasta', 'Sierra', 'Siskiyou', 'Solano', 'Sonoma', 'Stanislaus', 'Sutter', 'Tehama', 'Tulare', 'Tuolumne', 'Ventura', 'Yolo', 'Yuba']
Bay Area project rows: 1510
Missing unit counts: 72


In [ ]:
# HUD CHAS ZIP file
zip_path = "data/2018thru2022-050-csv (1).zip"
with zipfile.ZipFile(zip_path, "r") as z:
    with z.open("050/Table8.csv") as f:
        income_data = pd.read_csv(f, encoding="latin1")

# The nine Bay Area counties
bay_counties = [
    "Alameda", "Contra Costa", "Marin", "Napa", "San Francisco",
    "San Mateo", "Santa Clara", "Solano", "Sonoma"
]

income_data["county"] = income_data["name"].str.replace(
    " County, California", "", regex=False
)
bay_income = income_data[income_data["county"].isin(bay_counties)].copy()
bay_income["Below 50%"] = (
    bay_income["T8_est3"] + bay_income["T8_est16"]
    + bay_income["T8_est69"] + bay_income["T8_est82"]
)
bay_income["50–80%"] = (
    bay_income["T8_est29"] + bay_income["T8_est95"]
)

# Percentage of households below 80% of HAMFI
bay_income["Percent below 80%"] = (
    (bay_income["Below 50%"] + bay_income["50–80%"])
    / bay_income["T8_est1"] * 100
)

county_url = (
    "https://www2.census.gov/geo/tiger/GENZ2023/shp/"
    "cb_2023_us_county_500k.zip"
)
counties = gpd.read_file(county_url).to_crs("EPSG:3310")

bay_map = counties[
    (counties["STATEFP"] == "06") & (counties["NAME"].isin(bay_counties))
].copy()

bay_map = bay_map.merge(
    bay_income[["county", "Percent below 80%"]],
    left_on="NAME",
    right_on="county",
    how="left"
)